# 05 — HMM + Viterbi route reconstruction (Person B)

Classifier-only (each GPS point takes its highest-probability candidate) vs classifier + HMM/Viterbi
(one coherent route through the road network).

* **Model:** `artifacts/models/final_model.joblib` if it exists, otherwise the **Phase 1 random forest**
  (`data/models/random_forest.joblib`, 5 distance-only features). The cell below prints which one is used.
* **Parameter search:** 15 TRAIN trips (full-rate points), scored **out-of-fold** (each trip by a forest
  refitted without its CV fold), so the HMM is not tuned on over-confident in-sample probabilities.
* **Evaluation:** student_test, once, with the chosen parameters. Evaluator data is never used.
* **Labels** (route v2) are used only to score predictions, never as inputs.

Terminal version of the same steps: `python scripts/run_hmm_experiment.py`.

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
from IPython.display import Image, display
from config import RANDOM_STATE
from src.data.loader import list_trips
from src.hmm.viterbi import DEFAULT_PARAMS, decode_trip
from src.hmm.experiment import (load_scorer, load_trip_for_hmm, score_trips, param_search, compare,
                                error_runs, plot_trip_comparison)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
RES = ROOT / "results"
scorer = load_scorer()

## 1. Parameter search on 15 train trips (out-of-fold probabilities)

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
picked = sorted(rng.choice(list_trips("train"), size=15, replace=False).tolist())
print("train trips:", picked)
train_trips = score_trips([load_trip_for_hmm("train", t, scorer) for t in picked], scorer, out_of_fold=True)
search, params = param_search(train_trips)
search.insert(0, "model", scorer["source"]); search.insert(1, "oof_scores", True)
search.to_csv(RES / "hmm_param_search.csv", index=False)
(RES / "hmm_chosen_params.json").write_text(json.dumps({"params": params, "model": scorer["source"], "searched": True}, indent=2))
print("chosen:", params)
search.sort_values("top1", ascending=False).head(10)

## 2. student_test: classifier-only vs HMM (chosen parameters, evaluated once)

In [ ]:
test_trips = score_trips([load_trip_for_hmm("student_test", t, scorer) for t in list_trips("student_test")], scorer)
comp = compare(test_trips, params)
comp.insert(1, "model", scorer["source"])
comp.to_csv(RES / "hmm_comparison.csv", index=False)
comp[["trip_id", "n_points", "n_labelled", "top1_clf", "top1_hmm", "cont_clf", "cont_hmm",
      "reverse_err_clf", "reverse_err_hmm", "pct_points_changed_by_hmm"]]

## 3. Where the HMM helped most / least

In [ ]:
per = comp[comp["trip_id"] != "OVERALL"].copy()
per["top1_gain"] = per["top1_hmm"] - per["top1_clf"]
helped = per.sort_values("top1_gain", ascending=False)["trip_id"].head(2).tolist()
least = per.sort_values("top1_gain")["trip_id"].head(1).tolist()
worst = per.sort_values("top1_hmm")["trip_id"].head(1).tolist()
print("helped most:", helped, "| helped least:", least, "| lowest HMM Top-1:", worst)
per.sort_values("top1_gain", ascending=False)[["trip_id", "top1_clf", "top1_hmm", "top1_gain", "cont_clf", "cont_hmm",
    "reverse_err_clf", "reverse_err_hmm", "n_segments", "mean_candidates", "pct_points_changed_by_hmm"]]

In [ ]:
by_id = {d["trip_id"]: d for d in test_trips}
for tid in dict.fromkeys(least + worst):
    d = by_id[tid]
    runs, n_wrong, n_rev = error_runs(d, decode_trip(d["cand"], d["proba"], d["arcs"], d["clean"], params))
    print(f"trip {tid}: {n_wrong} wrong labelled points, {n_rev} of them the reverse twin of the true arc")
    display(runs)

In [ ]:
for tid in dict.fromkeys(helped + least + worst):
    display(Image(str(plot_trip_comparison(by_id[tid], params, RES / "figures" / f"hmm_student_test_{tid}.png"))))